# GMW v4 — Change Rasters: Band-1 Stack → GEE Upload

**Purpose**: For each of the 40 base-year archives in `GMW_extent_v4/change_rasters/`, extract Band 1 (the year immediately following the base year) from every geographic tile, and write it as one band of a single 40-band GeoTIFF, which is then uploaded to Google Earth Engine as a single multi-band Image asset.

**Band naming**: `b1` … `b40`, corresponding to base years 1985–2024 in ascending order.

## Workflow

```
Setup (first year only):
  1. Extract first year's tiles, build Band-1 VRT → determine output dimensions
  2. Open the final 40-band output file for writing

For each base year (1985 … 2024):
  3. Extract ~1,696 tiles from the .tar.gz into intermediate_files/tiles_{year}/
  4. For each tile, gdal_translate -b 1 → single-band tile (DEFLATE)
  5. gdalbuildvrt → VRT mosaic (a few KB text file, no GeoTIFF written)
  6. Write VRT data windowed into band N of the output file
  7. Delete tiles_{year}/ immediately

  Peak disk per year: ~200–500 MB (Band-1 tiles) + growing output file.
  No intermediate per-year GeoTIFF mosaics are created.

After all years:
  8. Upload the 40-band raster to Google Cloud Storage
  9. Ingest into GEE via manifest (single Image, 40 bands named b1–b40)
```

## Prerequisites

- `uv` environment with dependencies from `data/pyproject.toml` (rasterio, earthengine-api, google-cloud-storage)
- `gdalbuildvrt` and `gdal_translate` available on PATH (from GDAL install)
- Google Cloud SDK authenticated: `gcloud auth application-default login --project=mangrove-atlas-246414`
- Earth Engine authenticated: run `ee.Authenticate()` cell below

## 0. Configuration

> **Edit this cell** before running the notebook: set `GEE_ASSET_PATH` to the desired destination asset.

In [38]:
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths  (derived from notebook location — no hard-coded absolute paths)
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()                         # .../data/notebooks/Lab/data_processing
DATA_ROOT    = NOTEBOOK_DIR.parents[2]            # .../data
ARCHIVE_DIR  = DATA_ROOT / 'data' / 'GMW_extent_v4' / 'change_rasters'

INTERMEDIATE_DIR = ARCHIVE_DIR / 'intermediate_files'
TILES_DIR        = INTERMEDIATE_DIR / 'tiles'    # subdirs tiles/tiles_{year} — deleted per year
MOSAICS_DIR      = INTERMEDIATE_DIR / 'mosaics'  # one compressed GeoTIFF per year (~50 MB each)
STACK_VRT        = INTERMEDIATE_DIR / 'stack.vrt'
FINAL_RASTER     = INTERMEDIATE_DIR / 'gmw_change_stack_v4.tif'

# ---------------------------------------------------------------------------
# Google Cloud Storage
# ---------------------------------------------------------------------------
GCS_BUCKET      = 'mangrove_atlas'
GCS_DESTINATION = 'ee_import_data/gmw_change_stack/gmw_change_stack_v4.tif'

# ---------------------------------------------------------------------------
# Google Earth Engine
# ---------------------------------------------------------------------------
GEE_PROJECT    = 'mangrove-atlas-246414'
# ⚠️  Set the full asset path before running Step 4:
GEE_ASSET_PATH = f'projects/{GEE_PROJECT}/assets/land-cover/mangrove_change_stack_v4'

# ---------------------------------------------------------------------------
# Raster settings
# ---------------------------------------------------------------------------
NODATA_VALUE = 0
PYRAMIDING   = 'MODE'   # classification data — use MODE, not MEAN

print('Archive dir :', ARCHIVE_DIR)
print('Intermediate:', INTERMEDIATE_DIR)
print('Mosaics dir :', MOSAICS_DIR)
print('Final raster:', FINAL_RASTER)
print('GEE asset   :', GEE_ASSET_PATH)

Archive dir : /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4/change_rasters
Intermediate: /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4/change_rasters/intermediate_files
Mosaics dir : /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4/change_rasters/intermediate_files/mosaics
Final raster: /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4/change_rasters/intermediate_files/gmw_change_stack_v4.tif
GEE asset   : projects/mangrove-atlas-246414/assets/land-cover/mangrove_change_stack_v4


## 1. Imports & authentication

In [39]:
import json
import os
import re
import shutil
import subprocess
import tarfile
from typing import List

import numpy as np
import rasterio
from rasterio.crs import CRS

try:
    from google.cloud import storage as gcs
except ImportError:
    print('google-cloud-storage not found — install with: pip install google-cloud-storage')
    gcs = None

import ee

In [40]:
# Authenticate and initialise Earth Engine
# Only needed once per session; comment out after first run.
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

## 2. Discover archives

Collect all archives and sort by base year.

In [41]:
YEAR_RE = re.compile(r'base(\d{4})\.tar\.gz$')

archives = sorted(
    ARCHIVE_DIR.glob('gmw_mng_ext_v4112_stack_chng_cls_base*.tar.gz'),
    key=lambda p: int(YEAR_RE.search(p.name).group(1))
)

base_years = [int(YEAR_RE.search(p.name).group(1)) for p in archives]

print(f'Found {len(archives)} archives: {base_years[0]}–{base_years[-1]}')
assert len(archives) > 0, 'No archives found — check ARCHIVE_DIR'

# Create output directories
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
TILES_DIR.mkdir(parents=True, exist_ok=True)
MOSAICS_DIR.mkdir(parents=True, exist_ok=True)

Found 40 archives: 1985–2024


## 3. Per-year processing: extract Band 1 → single-band mosaic

For each archive:
1. Extract all `.tif` tiles into `intermediate_files/tiles/tiles_{year}/`
2. For every tile, copy only Band 1 → `{tile}_b1.tif` (original deleted immediately)
3. Build a VRT mosaic from the Band-1 tiles (~KB text file)
4. `gdal_translate` the VRT → compressed single-band GeoTIFF in `mosaics/`
5. Delete the tile directory

> **Disk peak during this phase**: ~200–500 MB (one year's tiles) + growing `mosaics/` directory (~50 MB/year, ~2 GB for all 40).
> Already-created mosaics are detected and skipped automatically — the loop is resumable.

In [42]:
def run(cmd: List[str]) -> None:
    """Run a shell command, raise on non-zero exit."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}\n{result.stderr}')


def extract_b1_tiles(archive_path: Path, year: int) -> tuple[Path, Path]:
    """
    Extract all tiles from archive, translate each to single-band (Band 1),
    delete the originals. Returns (tile_dir, b1_dir).
    """
    tile_dir = TILES_DIR / f'tiles_{year}'
    tile_dir.mkdir(parents=True, exist_ok=True)

    print(f'  [{year}] Extracting tiles …', end=' ', flush=True)
    with tarfile.open(archive_path, 'r:gz') as tf:
        tf.extractall(tile_dir, filter='data')
    # rglob because the archive may extract into a subdirectory.
    # b1_dir is created AFTER this call so it won't appear in results.
    orig_tiles = sorted(tile_dir.rglob('*.tif'))
    print(f'{len(orig_tiles)} tiles.')

    b1_dir = tile_dir / 'b1'
    b1_dir.mkdir(exist_ok=True)

    print(f'  [{year}] Extracting Band 1 …', end=' ', flush=True)
    for tile in orig_tiles:
        run([
            'gdal_translate', '-b', '1',
            '-a_nodata', str(NODATA_VALUE),
            '-co', 'COMPRESS=DEFLATE', '-co', 'TILED=YES',
            str(tile), str(b1_dir / tile.name),
        ])
        tile.unlink()  # free the multi-band tile immediately
    b1_tiles = sorted(b1_dir.glob('*.tif'))
    print(f'{len(b1_tiles)} Band-1 tiles.')
    return tile_dir, b1_dir


def build_vrt(b1_dir: Path, year: int) -> Path:
    """Build a lightweight VRT mosaic from Band-1 tiles; returns VRT path."""
    vrt_path = b1_dir.parent / f'mosaic_{year}.vrt'
    b1_tiles = sorted(b1_dir.glob('*.tif'))
    run(['gdalbuildvrt', '-vrtnodata', str(NODATA_VALUE), str(vrt_path)]
        + [str(t) for t in b1_tiles])
    return vrt_path


def mosaic_year_to_tif(archive_path: Path, year: int) -> Path:
    """
    Full pipeline for one base year: extract → Band-1 tiles → VRT → compressed GeoTIFF.
    Tile directory is deleted after translation. Returns mosaic GeoTIFF path.
    """
    mosaic_path = MOSAICS_DIR / f'mosaic_{year}.tif'
    if mosaic_path.exists():
        print(f'  [{year}] Mosaic already exists — skipping.')
        return mosaic_path

    tile_dir, b1_dir = extract_b1_tiles(archive_path, year)
    vrt = build_vrt(b1_dir, year)

    print(f'  [{year}] Writing mosaic GeoTIFF …', end=' ', flush=True)
    run([
        'gdal_translate',
        '-co', 'COMPRESS=DEFLATE', '-co', 'TILED=YES', '-co', 'BIGTIFF=IF_SAFER',
        str(vrt), str(mosaic_path),
    ])
    print(f'done — {mosaic_path.stat().st_size / 1e6:.1f} MB')

    shutil.rmtree(tile_dir)
    print(f'  [{year}] Tile directory deleted.')
    return mosaic_path

In [44]:
# ⚠️  This loop is the most time- and disk-intensive part of this section.
# Each archive takes several minutes. Already-created mosaics are skipped automatically.

mosaic_paths = []

for archive, year in zip(archives, base_years):
    print(f'\nProcessing base year {year} — {archive.name}')
    tile_dir = TILES_DIR / f'tiles_{year}'
    if tile_dir.exists():
        shutil.rmtree(tile_dir)
    mp = mosaic_year_to_tif(archive, year)
    mosaic_paths.append(mp)

print(f'\n✓ All {len(mosaic_paths)} year mosaics ready in {MOSAICS_DIR}')


Processing base year 1985 — gmw_mng_ext_v4112_stack_chng_cls_base1985.tar.gz
  [1985] Extracting tiles … 1696 tiles.
  [1985] Extracting Band 1 … 1696 Band-1 tiles.
  [1985] Writing mosaic GeoTIFF … done — 537.9 MB
  [1985] Tile directory deleted.

Processing base year 1986 — gmw_mng_ext_v4112_stack_chng_cls_base1986.tar.gz
  [1986] Extracting tiles … 1696 tiles.
  [1986] Extracting Band 1 … 1696 Band-1 tiles.
  [1986] Writing mosaic GeoTIFF … done — 538.3 MB
  [1986] Tile directory deleted.

Processing base year 1987 — gmw_mng_ext_v4112_stack_chng_cls_base1987.tar.gz
  [1987] Extracting tiles … 1696 tiles.
  [1987] Extracting Band 1 … 1696 Band-1 tiles.
  [1987] Writing mosaic GeoTIFF … done — 538.6 MB
  [1987] Tile directory deleted.

Processing base year 1988 — gmw_mng_ext_v4112_stack_chng_cls_base1988.tar.gz
  [1988] Extracting tiles … 1696 tiles.
  [1988] Extracting Band 1 … 1696 Band-1 tiles.
  [1988] Writing mosaic GeoTIFF … done — 540.3 MB
  [1988] Tile directory deleted.

Pro

KeyboardInterrupt: 

## 4. Stack year mosaics → 40-band output

`gdalbuildvrt -separate` builds a lightweight multi-band VRT (one band per year mosaic).
`gdal_translate` then writes the final 40-band compressed GeoTIFF.
`CHECK_DISK_FREE_SPACE=FALSE` bypasses GDAL's conservative uncompressed-size pre-check
(~147 GB); the actual DEFLATE-compressed output is ~1–2 GB.

In [33]:
mosaic_files = sorted([MOSAICS_DIR / f'mosaic_{y}.tif' for y in base_years])
missing = [m for m in mosaic_files if not m.exists()]
if missing:
    raise FileNotFoundError(f'Missing mosaics: {[m.name for m in missing]}')

# 1. Build a stacked VRT — one band per year mosaic
print(f'Building stacked VRT from {len(mosaic_files)} mosaics …', end=' ', flush=True)
run(['gdalbuildvrt', '-separate', str(STACK_VRT)] + [str(m) for m in mosaic_files])
print('done.')

# 2. Translate stacked VRT → final 40-band compressed GeoTIFF
# --config CHECK_DISK_FREE_SPACE FALSE skips the uncompressed-size pre-check
print(f'Writing final {len(mosaic_files)}-band raster (this will take a while) …')
run([
    'gdal_translate',
    '--config', 'CHECK_DISK_FREE_SPACE', 'FALSE',
    '-co', 'COMPRESS=DEFLATE',
    '-co', 'TILED=YES',
    '-co', 'BIGTIFF=YES',
    str(STACK_VRT),
    str(FINAL_RASTER),
])
print(f'✓ Final raster written: {FINAL_RASTER}')
print(f'  Size on disk: {FINAL_RASTER.stat().st_size / 1e9:.2f} GB')

# 3. Tag each band with base year and band name
with rasterio.open(FINAL_RASTER, 'r+') as ds:
    for i, year in enumerate(base_years, start=1):
        ds.update_tags(i, base_year=str(year), band_name=f'b{i}')
print('✓ Band metadata written.')

Building stacked VRT from 40 mosaics … done.
Writing final 40-band raster (this will take a while) …
✓ Final raster written: /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4/change_rasters/intermediate_files/gmw_change_stack_v4.tif
  Size on disk: 15.11 GB
✓ Band metadata written.


In [34]:
# Verify band count, CRS, and per-band metadata before uploading
with rasterio.open(FINAL_RASTER) as ds:
    print(f'Bands : {ds.count}')
    print(f'Size  : {ds.width} × {ds.height} px')
    print(f'CRS   : {ds.crs}')
    print(f'Dtype : {ds.dtypes[0]}')
    print(f'NoData: {ds.nodata}')
    for i in range(1, ds.count + 1):
        print(f'  Band {i:2d}: {ds.tags(i)}')

Bands : 40
Size  : 1335960 × 274614 px
CRS   : GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST]]
Dtype : uint8
NoData: 0.0
  Band  1: {'band_name': 'b1', 'base_year': '1985'}
  Band  2: {'band_name': 'b2', 'base_year': '1986'}
  Band  3: {'band_name': 'b3', 'base_year': '1987'}
  Band  4: {'band_name': 'b4', 'base_year': '1988'}
  Band  5: {'band_name': 'b5', 'base_year': '1989'}
  Band  6: {'band_name': 'b6', 'base_year': '1990'}
  Band  7: {'band_name': 'b7', 'base_year': '1991'}
  Band  8: {'band_name': 'b8', 'base_year': '1992'}
  Band  9: {'band_name': 'b9', 'base_year': '1993'}
  Band 10: {'band_name': 'b10', 'base_year': '1994'}
  Band 11: {'band_name': 'b11', 'base_year': '1995'}
  Band 12: {'band_name': 'b12', 'base_year': '1996'}
  Band 13: {'band_name': 'b13', 'base_year': '1997'}
  Band 14: {'band_name': '

In [ ]:
# Delete year mosaics and the stack VRT once the final raster is verified
for mp in mosaic_files:
    mp.unlink(missing_ok=True)
STACK_VRT.unlink(missing_ok=True)
shutil.rmtree(MOSAICS_DIR, ignore_errors=True)
print('✓ Year mosaics and stack VRT deleted.')

## 5. Upload to Google Cloud Storage

The 40-band raster is uploaded to the `mangrove_atlas` GCS bucket. Ensure you are authenticated:

```bash
gcloud auth application-default login --project=mangrove-atlas-246414
```

## 6. Create GEE manifest and ingest as Image asset

The manifest tells GEE:
- Source: the single 40-band GeoTIFF in GCS
- Bands: `b1`–`b40`, mapped to GeoTIFF band indices 0–39
- Pyramiding: `MODE` (classification data)
- NoData: 0

The `earthengine upload image` CLI is used because it supports manifest-based multi-band ingestion.

In [35]:
n_bands_final = len(base_years)

manifest = {
    'name': GEE_ASSET_PATH,
    'tilesets': [
        {
            'id': 'change_stack',
            'sources': [
                {'uris': [f'gs://{GCS_BUCKET}/{GCS_DESTINATION}']}
            ],
        }
    ],
    'bands': [
        {
            'id': f'b{i}',
            'tileset_id': 'change_stack',
            'tileset_band_index': i - 1,
        }
        for i in range(1, n_bands_final + 1)
    ],
    'missing_data': {'values': [NODATA_VALUE]},
    'pyramiding_policy': PYRAMIDING,
    'properties': {
        'version': 'v4',
        'band_description': ', '.join(
            f'b{i}=base{year}' for i, year in enumerate(base_years, start=1)
        ),
    },
}

# Write manifest alongside the final raster
MANIFEST_PATH = INTERMEDIATE_DIR / 'gee_manifest.json'
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Manifest written to {MANIFEST_PATH}')
print(f'Bands defined: {n_bands_final} (b1 … b{n_bands_final})')

Manifest written to /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4/change_rasters/intermediate_files/gee_manifest.json
Bands defined: 40 (b1 … b40)


In [36]:
# Check whether the destination asset already exists; delete if so.
asset_info = None
try:
    asset_info = ee.data.getAsset(GEE_ASSET_PATH)
    print(f'Asset already exists: {GEE_ASSET_PATH}')
    print('Deleting existing asset before re-upload …')
    ee.data.deleteAsset(GEE_ASSET_PATH)
    print('Deleted.')
except ee.ee_exception.EEException:
    print('No existing asset found — proceeding with fresh upload.')

No existing asset found — proceeding with fresh upload.


In [37]:
# Start the GEE ingestion task
result = subprocess.run(
    ['earthengine', '--project', GEE_PROJECT,
     'upload', 'image', '--manifest', str(MANIFEST_PATH)],
    capture_output=True, text=True,
)

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('earthengine upload failed — see output above')

print('✓ Ingestion task submitted.')

Started upload task with ID: H6NVZWOYYRGGN6I2LPKECDZM

✓ Ingestion task submitted.


## 7. Monitor ingestion task

In [ ]:
# List running tasks to confirm the task was registered
result = subprocess.run(
    ['earthengine', '--project', GEE_PROJECT, 'task', 'list', '--status', 'RUNNING'],
    capture_output=True, text=True,
)
print(result.stdout or '(no running tasks — may have already completed or failed)')

In [ ]:
# Once the task shows COMPLETED, verify the asset
asset = ee.data.getAsset(GEE_ASSET_PATH)
print('Asset type   :', asset.get('type'))
print('Asset path   :', asset.get('name'))

img = ee.Image(GEE_ASSET_PATH)
print('Band names   :', img.bandNames().getInfo())
print('Projection   :', img.select('b1').projection().getInfo())

## 8. (Optional) Clean up local intermediates

After successful GEE ingestion, the local 40-band raster can be deleted to free disk space. The manifest JSON is kept as a record.

In [ ]:
# Uncomment and run only after confirming the GEE asset is correct

# FINAL_RASTER.unlink(missing_ok=True)
# print(f'Deleted local raster: {FINAL_RASTER}')

---

## Manual steps if GEE ingestion cannot be automated

If the `earthengine` CLI is not authenticated or `ee.Authenticate()` fails in this environment, follow these steps:

### A. Authenticate (terminal)

```bash
# Earth Engine
earthengine authenticate

# Google Cloud (for GCS upload)
gcloud auth application-default login --project=mangrove-atlas-246414
```

### B. Upload to GCS (terminal)

```bash
gsutil -m cp \
  data/data/GMW_extent_v4/change_rasters/intermediate_files/gmw_change_stack_v4.tif \
  gs://mangrove_atlas/ee_import_data/gmw_change_stack/gmw_change_stack_v4.tif
```

### C. Ingest to GEE via manifest (terminal)

```bash
earthengine --project mangrove-atlas-246414 \
  upload image \
  --manifest data/data/GMW_extent_v4/change_rasters/intermediate_files/gee_manifest.json
```

### D. Check task status

```bash
earthengine task list --status RUNNING
```

### E. Verify asset in GEE Code Editor

```javascript
var img = ee.Image('projects/mangrove-atlas-246414/assets/land-cover/mangrove_change_stack_v4');
print(img.bandNames());
print(img.projection());
Map.addLayer(img.select('b1'), {min: 0, max: 4, palette: ['white','green','red','orange','blue']}, 'Band 1 (base 1985)');
```

---

## 9. Alternative: Ingest year mosaics as ImageCollection

Instead of a single 40-band Image, ingest each year mosaic as an individual GEE Image inside an ImageCollection. This is the preferred structure for time-series calculations (filter by year, map over the collection, use `frequencyHistogram` for efficient gain/loss area computation in a single reducer pass).

**When to use this approach instead of §5–§6:**
- You want to filter or map over years in GEE without selecting bands by name
- You plan to use `frequencyHistogram` to decode classification values into gain/loss area
- You want to extend the dataset with new years without re-ingesting everything

**Prerequisites:**
- Year mosaics already uploaded to GCS at  
  `gs://mangrove_atlas/ee_import_data/gmw_change_stack/gain_loss_mosaics/mosaic_{year}.tif`
- Cells §0–§2 have been run (`base_years`, `GEE_PROJECT`, `NODATA_VALUE`, `PYRAMIDING` are defined)

In [45]:
# ---------------------------------------------------------------------------
# Configuration for ImageCollection ingestion
# ---------------------------------------------------------------------------

# GCS prefix where the year mosaics have been uploaded (no trailing slash)
GCS_MOSAICS_PREFIX = 'ee_import_data/gmw_change_stack/gain_loss_mosaics'

# Destination ImageCollection asset path
GEE_IC_ASSET_PATH = f'projects/{GEE_PROJECT}/assets/land-cover/mangrove_change_ic_v4'

# Directory to write per-year manifest files
IC_MANIFESTS_DIR = INTERMEDIATE_DIR / 'ic_manifests'
IC_MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

print('GCS mosaics prefix:', f'gs://{GCS_BUCKET}/{GCS_MOSAICS_PREFIX}/')
print('GEE IC asset      :', GEE_IC_ASSET_PATH)
print('Manifests dir     :', IC_MANIFESTS_DIR)

GCS mosaics prefix: gs://mangrove_atlas/ee_import_data/gmw_change_stack/gain_loss_mosaics/
GEE IC asset      : projects/mangrove-atlas-246414/assets/land-cover/mangrove_change_ic_v4
Manifests dir     : /Users/angel/Documents/REPOSITORIOS/mangrove-atlas/data/data/GMW_extent_v4/change_rasters/intermediate_files/ic_manifests


### 9b. Create ImageCollection asset and ingest images

Once all mosaics are in GCS, create the ImageCollection container and submit one ingestion task per year.

Each Image's `system:time_start` is set to **year + 1** (e.g. `mosaic_1985` → `1986-01-01`) because Band 1 of the base-year archive records the change that occurred in the following year.

In [46]:
# Create the ImageCollection asset container.
# GEE requires the collection to exist before images can be ingested into it.
try:
    ee.data.createAsset({'type': 'ImageCollection'}, GEE_IC_ASSET_PATH)
    print(f'✓ Created ImageCollection: {GEE_IC_ASSET_PATH}')
except ee.ee_exception.EEException as exc:
    if 'already exists' in str(exc).lower():
        print(f'ImageCollection already exists: {GEE_IC_ASSET_PATH}')
    else:
        raise

✓ Created ImageCollection: projects/mangrove-atlas-246414/assets/land-cover/mangrove_change_ic_v4


In [47]:
# Generate one manifest per year and submit ingestion tasks.
# system:time_start = year+1 (the year the change was measured).
# system:time_end   = year+2 (exclusive end of the 1-year window).

task_ids = {}

for year in base_years:
    gcs_uri  = f'gs://{GCS_BUCKET}/{GCS_MOSAICS_PREFIX}/mosaic_{year}.tif'
    asset_id = f'{GEE_IC_ASSET_PATH}/mosaic_{year}'

    manifest = {
        'name': asset_id,
        'tilesets': [
            {
                'id': f'change_{year}',
                'sources': [{'uris': [gcs_uri]}],
            }
        ],
        'bands': [
            {
                'id': 'classification',
                'tileset_id': f'change_{year}',
                'tileset_band_index': 0,
            }
        ],
        'missing_data': {'values': [NODATA_VALUE]},
        'pyramiding_policy': PYRAMIDING,
        'start_time': f'{year}-01-01T00:00:00Z',
        'end_time':   f'{year + 1}-01-01T00:00:00Z',
        'properties': {
            'base_year':        year,
            'change_year':      year + 1,
            'version':          'v4',
        },
    }

    manifest_path = IC_MANIFESTS_DIR / f'manifest_{year}.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    result = subprocess.run(
        ['earthengine', '--project', GEE_PROJECT,
         'upload', 'image', '--manifest', str(manifest_path)],
        capture_output=True, text=True,
    )

    if result.returncode != 0:
        print(f'[{year}] FAILED: {result.stderr.strip()}')
    else:
        task_id = result.stdout.strip().split()[-1]
        task_ids[year] = task_id
        print(f'[{year}] Submitted — task {task_id}')

print(f'\n✓ {len(task_ids)}/{len(base_years)} ingestion tasks submitted.')

[1985] Submitted — task ZRD3FLG6GFHU3UOXWB4SUYQ6
[1986] Submitted — task CTHJH2PIXTO4BTOQJ2Y4UEKD
[1987] Submitted — task BTES2353TKVZT3C5B7C33MCV
[1988] Submitted — task FSBIYH7MXSMRLHNTWHJDCRIX
[1989] Submitted — task GF345Y4DWJGYFYBPABYNLPO3
[1990] Submitted — task 5GMIB4C3ZSA5PK2J3XMHLIPB
[1991] Submitted — task 5TT46ZTX2FZ6QK3NZ27QXBBC
[1992] Submitted — task YGVGSN5KELJU3HVAZ27EMTVN
[1993] Submitted — task IKBV2B2DLVOMYRAN7GHNMNXO
[1994] Submitted — task VADLXGZ677XA6BQ4LQWG2ZAX
[1995] Submitted — task FH3OUQHFPLZLGNS5J4A5KJJW
[1996] Submitted — task GCYZQD7GPUV2LBULFMO3KIQX
[1997] Submitted — task YLIGVRRHN2RPRSLECPJVTUKS
[1998] Submitted — task 2UYPPG6F2ZKOFBOJOXPFKZJX
[1999] Submitted — task FURAHBO2Z3LU4I5AXURXSDOH
[2000] Submitted — task LF7C37GE5JQH72L4T3XVRTX4
[2001] Submitted — task PSQ5YFYLYNMPZMT7Q3KYD2SW
[2002] Submitted — task WA5ZUNV7RCDQXBEN3IXTUBKE
[2003] Submitted — task NP6GB3UIKI3Q5MPQMNDAFBE4
[2004] Submitted — task OCSDZHG5N3ZCOH7PEBVVAAKR
[2005] Submitted — t

In [48]:
# Monitor running ingestion tasks. Re-run periodically until all complete.
result = subprocess.run(
    ['earthengine', '--project', GEE_PROJECT, 'task', 'list', '--status', 'RUNNING'],
    capture_output=True, text=True,
)
print(result.stdout.strip() or '(no running tasks — all may have completed or failed)')

result_failed = subprocess.run(
    ['earthengine', '--project', GEE_PROJECT, 'task', 'list', '--status', 'FAILED'],
    capture_output=True, text=True,
)
if result_failed.stdout.strip():
    print('\nFAILED tasks:')
    print(result_failed.stdout.strip())

NP6GB3UIKI3Q5MPQMNDAFBE4  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
WA5ZUNV7RCDQXBEN3IXTUBKE  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
PSQ5YFYLYNMPZMT7Q3KYD2SW  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
LF7C37GE5JQH72L4T3XVRTX4  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
FURAHBO2Z3LU4I5AXURXSDOH  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
2UYPPG6F2ZKOFBOJOXPFKZJX  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
YLIGVRRHN2RPRSLECPJVTUKS  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
GCYZQD7GPUV2LBULFMO3KIQX  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
FH3OUQHFPLZLGNS5J4A5KJJW  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
VADLXGZ677XA6BQ4LQWG2ZAX  Upload        Ingest image: "projects/mangrove-atlas-2..  RUNNING    ---
IKBV2B2DLV

In [ ]:
# Verify the ImageCollection once all tasks complete.
ic = ee.ImageCollection(GEE_IC_ASSET_PATH)
count = ic.size().getInfo()
print(f'Images in collection : {count} (expected {len(base_years)})')

change_years = sorted(ic.aggregate_array('change_year').getInfo())
print(f'Change years         : {change_years[0]}–{change_years[-1]}')

first = ic.first()
print(f'Band names (first)   : {first.bandNames().getInfo()}')
print(f'Projection (first)   : {first.select("classification").projection().getInfo()}')

missing = sorted(set(y + 1 for y in base_years) - set(change_years))
if missing:
    print(f'\n⚠️  Missing change years: {missing}')
else:
    print(f'\n✓ All {count} images verified.')